### Funkcje pomocnicze

In [ ]:
def podziel_liste( L ) :
    """
    Funkcja zwraca dwie listy E oraz O, składające się z elementów listy L,
    mających odpowiednio parzyste i nieparzyste indeksy.
    """
    n = len(L)
    E = [ L[j] for j in [0,2,..,n-1] ]
    O = [ L[j] for j in [1,3,..,n-1] ]
    return E,O


### Zadania/Exercises

<p><strong>Zadanie:</strong> Napisać funkcję, która oblicza pierwiastek pierwotny z jedynki podanego stopnia.</p>

In [ ]:
def pierw_pierw( n, K = CC ) :
    """
    Funkcja wyznacza pierwiastek pierwotny z 1 stopnia n w ciele K.
    """
    return K( cos(2*pi/n) + i*sin(2*pi/n) )

In [ ]:
l = [1..8]; show("l = "+latex(l))
omega = pierw_pierw( len(l), SR )
L = [ sum( l[k]*omega^(-k*n) for k in range(len(l)) ) for n in range(len(l)) ]
show("\\hat{l}="+latex(L))

l = \left[1, 2, 3, 4, 5, 6, 7, 8\right]

\hat{l}= \left[36, 4 i \, \sqrt{2} + 4 i - 4, 4 i - 4, 4 i \, \sqrt{2} - 4 i - 4, -4, -4 i \, \sqrt{2} + 4 i - 4, -4 i - 4, -4 i \, \sqrt{2} - 4 i - 4\right]

<p><strong>Zadanie:</strong> Zaimplementować algorytm Cooley'a-Tukey'a szybkiej transformaty Fouriera.</p>

In [ ]:
def fft( L, omega = 0 ) :
    """
    Funkcja oblicza FFT listy L, używając algorytmu Cooley-Tukey. 
    Długość listy musi być potęgą 2
    
    Argumenty:
        
        - L - lista
        
        - omega - pierwiastek pierwotny z jedynki stopnia len( L ).
        Można go pominąć przy wywołaniu funkcji. Zostanie wtedy obliczony w locie.
    """
    
    # odczytujemy długość listy L
    n = len(L)
    # warunek stopu rekursji = lista 1-elementowa
    if n == 1 :
        return L
    assert NN(n).is_power_of(2), "Długość L musi być potęgą 2"
    # wyznaczamy pierwiastek pierwotny z jedynki (o ile nie został podany)
    if omega == 0:
        omega = pierw_pierw(n)

    # konstruujemy części parzyste i nieparzyste
    e, o = podziel_liste(L)

    # rekurencyjnie obliczamy ich transformaty Fouriera 
    E = fft(e, omega^2)
    O = fft(o, omega^2)

    # łączymy wyniki cząstkowe
    L0 = [ E[j] + omega^(-j)*O[j] for j in [0..n//2-1] ] 
    L1 = [ E[j] - omega^(-j)*O[j] for j in [0..n//2-1] ] 

    # zwracamy ostateczny wynik
    return L0 + L1

In [ ]:
# sprawdzenie
show(fft( [1..4] ))

[10.0000000000000,
 -2.00000000000000 + 2.00000000000000*I,
 -2.00000000000000,
 -2.00000000000000 - 2.00000000000000*I]

<p><strong>Zadanie:</strong> Napisać funkcję, która wyznacza kolejną potęgę dwójki, nie mniejszą od podanego argumenty. Ściśle mówiąc, dla liczby $ > 0$, funkcja zwraca $2^N$ t.ż. $2^{N-1} < n \leq 2^N$.</p>

In [ ]:
def nast_pot_2( n ) :
    """
    Funkcja wyznacza 2^N t.ż. 2^(N-1) < n <= 2^N
    """
    assert n > 0, "Liczba n musi być dodatnia!"
    N = ZZ(ceil(log(n, 2)))
    return 2^N

In [ ]:
# sprawdzenie
for j in [1..16] :
    print( j, '\t', nast_pot_2(j) )

1 	 1
2 	 2
3 	 4
4 	 4
5 	 8
6 	 8
7 	 8
8 	 8
9 	 16
10 	 16
11 	 16
12 	 16
13 	 16
14 	 16
15 	 16
16 	 16


<p><strong>Zadanie:</strong> Napisać funkcję obliczającą iloczyn dwóch wielomianów za pomocą transformaty Fouriera.</p>

In [ ]:
def mnozenie_fft( f, g ) :
    """
    Funkcja oblicza iloczyn dwóch wielomianów za pomocą transformaty Fouriera.
    """
    # pierścień wielomianów
    P = f.parent()
    # wyznaczamy długość wyniku
    N = 2*max( nast_pot_2(f.degree()), nast_pot_2(g.degree()) )
    # konwertujemy f, g do list o długości N
    f_ = f.list() + [0]*(N - f.degree() - 1)
    g_ = g.list() + [0]*(N - g.degree() - 1)
    # obliczamy fft obu list
    omega = pierw_pierw(N)
    F = fft(f_, omega)
    G = fft(g_, omega)
    # mnożymy wyraz po wyrazie
    H = [ F[j]*G[j] for j in [0..N-1] ]
    # obliczamy transformatę odwrotną
    h_ = [ hi/N for hi in fft(H, omega^(-1)) ]
    # zaokrąglamy do liczb całkowitych
    h_ = [ round(real_part(hi)) for hi in h_ ]
    # przekształcamy w wielomian
    return P(h_)

In [ ]:
P.<x> = ZZ[]
f = P.random_element(7); show("f = " + latex(f))
g = P.random_element(5); show("g = " + latex(g))

f = x^{7} - x^{6} - 5x^{5} - x^{4} - x^{3} + 3x^{2} - x - 2

g = -x^{5} + 2x^{3} - x^{2} - x + 1

In [ ]:
mnozenie_fft(f,g)

-x^12 + x^11 + 7*x^10 - 2*x^9 - 9*x^8 + 2*x^7 + 4*x^6 + 5*x^5 - 5*x^4 - 7*x^3 + 6*x^2 + x - 2

<p><strong>Zadanie:</strong> Napisać funkcję obliczającą iloczyn dwóch wielomianów za pomocą "mnożenia szkolnego". Porównać czasy działania obu funkcji dla wielomianów wysokich stopni (np. 5000).</p>

In [ ]:
def mnozenie_pisemne( f, g ) :
    """
    Funkcja oblicza iloczyn dwóch wielomianów metodą szkolną.
    """
    P = f.parent()
    m = f.degree()
    n = g.degree()
    h_ = [ sum( f[j]*g[k-j] for j in [max(0, k-n)..min(k,m)] ) for k in [0..m+n] ]
    return P(h_)

In [ ]:
f = P.random_element(5000)
g = P.random_element(5000)

In [ ]:
%%time
h = mnozenie_pisemne(f,g)

CPU times: user 40.4 s, sys: 32 ms, total: 40.4 s
Wall time: 40.5 s


In [ ]:
%%time
h = mnozenie_fft(f,g)

CPU times: user 6.26 s, sys: 8.01 ms, total: 6.27 s
Wall time: 6.27 s


In [ ]:
%%time
h = f*g

CPU times: user 780 µs, sys: 0 ns, total: 780 µs
Wall time: 783 µs
